# Experiment 0 — AfriSenti Kinyarwanda Data Exploration

This notebook loads the real-world AfriSenti Kinyarwanda dataset, verifies the splits and labels, inspects class balance and text quality, and records preprocessing decisions.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
for candidate in [ROOT / 'src', ROOT.parent / 'src', ROOT.parent.parent / 'src']:
    if (candidate / 'data_loader.py').exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError('Could not find src/data_loader.py. Open Jupyter from the project root.')

from data_loader import load_split


In [ ]:
splits = {name: load_split(name) for name in ['train', 'validation', 'test']}
for name, df in splits.items():
    print(f'{name}: {len(df):,} rows')
    print(df.head(3).to_string(index=False))
    print()


In [ ]:
for name, df in splits.items():
    print(name)
    print(df['label'].value_counts())
    print('Missing text:', int(df['tweet'].isna().sum()))
    print('Duplicate tweets:', int(df['tweet'].duplicated().sum()))
    print()


In [ ]:
train = splits['train'].copy()
train['char_len'] = train['tweet'].astype(str).str.len()
train['word_len'] = train['tweet'].astype(str).str.split().str.len()
display(train[['char_len', 'word_len']].describe())


In [ ]:
ax = sns.countplot(data=train, x='label', order=['negative', 'neutral', 'positive'])
ax.set_title('Kinyarwanda training-set class distribution')
ax.set_xlabel('Sentiment')
ax.set_ylabel('Number of tweets')
plt.tight_layout()
plt.show()


In [ ]:
# Check for identical tweets across dataset splits
tweet_sets = {
    name: set(df["tweet"].astype(str).str.strip())
    for name, df in splits.items()
}

pairs = [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test")
]

for first, second in pairs:
    overlap = tweet_sets[first] & tweet_sets[second]
    print(f"{first} vs {second}: {len(overlap)} shared tweets")

In [ ]:
for first, second in pairs:
    left = splits[first][["tweet", "label"]].copy()
    right = splits[second][["tweet", "label"]].copy()

    left["tweet"] = left["tweet"].astype(str).str.strip()
    right["tweet"] = right["tweet"].astype(str).str.strip()

    left = left.drop_duplicates()
    right = right.drop_duplicates()

    shared = left.merge(
        right,
        on="tweet",
        suffixes=(f"_{first}", f"_{second}")
    )

    label_conflicts = shared[
        shared[f"label_{first}"] != shared[f"label_{second}"]
    ]

    print(f"\n{first} vs {second}")
    print("Shared tweet texts:", shared["tweet"].nunique())
    print("Tweets with conflicting labels:",
          label_conflicts["tweet"].nunique())

    display(label_conflicts.drop_duplicates("tweet").head(3))

In [ ]:

from pathlib import Path
import re

# Normalize whitespace only; preserve the original tweet text.
def normalize_tweet(text):
    return re.sub(r"\s+", " ", str(text)).strip()

# Create independent copies and identify duplicate texts.
deduped = {}
within_split_removed = {}

for name, df in splits.items():
    temp = df[["tweet", "label"]].copy()
    temp["tweet_norm"] = temp["tweet"].map(normalize_tweet)

    before = len(temp)
    temp = temp.drop_duplicates(
        subset="tweet_norm",
        keep="first"
    )

    within_split_removed[name] = before - len(temp)
    deduped[name] = temp

# Keep the test set as the held-out evaluation set.
test_texts = set(deduped["test"]["tweet_norm"])

# Remove validation tweets that also occur in the test set.
validation_clean = deduped["validation"][
    ~deduped["validation"]["tweet_norm"].isin(test_texts)
].copy()

# Remove training tweets that occur in either held-out split.
heldout_texts = (
    set(deduped["validation"]["tweet_norm"])
    | test_texts
)

train_clean = deduped["train"][
    ~deduped["train"]["tweet_norm"].isin(heldout_texts)
].copy()

test_clean = deduped["test"].copy()

# Save processed copies; keep the original text and labels.
output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

clean_splits = {
    "train": train_clean,
    "validation": validation_clean,
    "test": test_clean
}

for name, df in clean_splits.items():
    df[["tweet", "label"]].to_csv(
        output_dir / f"{name}.tsv",
        sep="\t",
        index=False
    )

# Report exactly what changed.
print("Duplicate rows removed within each split:")
print(within_split_removed)

print("\nRows remaining after cross-split cleanup:")
for name, df in clean_splits.items():
    print(f"{name}: {len(df):,}")

print("\nRemaining shared tweet texts:")
for first, second in pairs:
    first_texts = set(clean_splits[first]["tweet"].map(normalize_tweet))
    second_texts = set(clean_splits[second]["tweet"].map(normalize_tweet))
    print(f"{first} vs {second}: {len(first_texts & second_texts)}")

print("\nCleaned copies saved in data/processed/")


In [ ]:
# Check repeated tweet texts for conflicting labels
for name, df in splits.items():
    temp = df.copy()
    temp["tweet_norm"] = temp["tweet"].map(normalize_tweet)

    labels_per_tweet = temp.groupby("tweet_norm")["label"].nunique()
    conflicts = labels_per_tweet[labels_per_tweet > 1]

    print(f"\n{name}")
    print("Repeated tweet texts with conflicting labels:", len(conflicts))

    if len(conflicts) > 0:
        examples = temp[
            temp["tweet_norm"].isin(conflicts.index)
        ][["tweet", "label"]].drop_duplicates()

        display(examples.head(10))

In [ ]:

from pathlib import Path
import re

def normalize_tweet(text):
    return re.sub(r"\s+", " ", str(text)).strip()

prepared = {}
conflict_summary = {}
duplicate_summary = {}

# First, handle conflicting labels within each split.
for name, df in splits.items():
    temp = df[["tweet", "label"]].copy()
    temp["tweet_norm"] = temp["tweet"].map(normalize_tweet)

    # Identify tweet texts assigned more than one sentiment.
    labels_per_tweet = temp.groupby("tweet_norm")["label"].nunique()
    conflicting_texts = set(labels_per_tweet[labels_per_tweet > 1].index)

    conflicting_rows = temp["tweet_norm"].isin(conflicting_texts)
    conflict_summary[name] = len(conflicting_texts)

    # Remove every occurrence of ambiguous tweet texts.
    temp = temp.loc[~conflicting_rows].copy()

    # Collapse repeated texts that have the same label.
    before = len(temp)
    temp = temp.drop_duplicates(subset="tweet_norm", keep="first")
    duplicate_summary[name] = before - len(temp)

    prepared[name] = temp

# Preserve the test set as the held-out evaluation set.
test_clean = prepared["test"].copy()
test_texts = set(test_clean["tweet_norm"])

# Prevent overlap between validation and test.
validation_clean = prepared["validation"][
    ~prepared["validation"]["tweet_norm"].isin(test_texts)
].copy()

# Prevent overlap between training and either held-out split.
heldout_texts = (
    set(validation_clean["tweet_norm"]) | test_texts
)

train_clean = prepared["train"][
    ~prepared["train"]["tweet_norm"].isin(heldout_texts)
].copy()

clean_splits = {
    "train": train_clean,
    "validation": validation_clean,
    "test": test_clean
}

# Save the cleaned copies.
output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

for name, df in clean_splits.items():
    df[["tweet", "label"]].to_csv(
        output_dir / f"{name}.tsv",
        sep="\t",
        index=False
    )

print("Conflicting tweet texts removed:", conflict_summary)
print("Same-label duplicate rows removed:", duplicate_summary)

print("\nFinal dataset sizes:")
for name, df in clean_splits.items():
    print(f"{name}: {len(df):,}")

print("\nFinal class distribution:")
for name, df in clean_splits.items():
    print(f"\n{name}")
    print(df["label"].value_counts())

print("\nShared tweet texts between splits:")
for first, second in pairs:
    a = set(clean_splits[first]["tweet_norm"])
    b = set(clean_splits[second]["tweet_norm"])
    print(f"{first} vs {second}: {len(a & b)}")

print("\nProcessed files saved successfully.")


In [ ]:

from pathlib import Path
import json
import joblib

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

# Experiment 1: TF-IDF + Logistic Regression
baseline_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )
    )
])

# Train on training data only.
baseline_model.fit(
    train_clean["tweet"].astype(str),
    train_clean["label"]
)

# Evaluate on validation data, not the held-out test set.
validation_predictions = baseline_model.predict(
    validation_clean["tweet"].astype(str)
)

y_true = validation_clean["label"]
y_pred = validation_predictions

accuracy = accuracy_score(y_true, y_pred)

precision, recall, macro_f1, _ = (
    precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )
)

print("EXPERIMENT 1: TF-IDF + LOGISTIC REGRESSION")
print("-" * 45)
print(f"Training examples: {len(train_clean):,}")
print(f"Validation examples: {len(validation_clean):,}")
print(f"Accuracy:          {accuracy:.4f}")
print(f"Macro precision:   {precision:.4f}")
print(f"Macro recall:      {recall:.4f}")
print(f"Macro F1:          {macro_f1:.4f}")

print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred,
    labels=["negative", "neutral", "positive"],
    target_names=["Negative", "Neutral", "Positive"],
    zero_division=0
))

# Plot the validation confusion matrix.
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=["negative", "neutral", "positive"]
)

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Negative", "Neutral", "Positive"]
)

display.plot(values_format="d")
plt.title("Experiment 1 — Validation Confusion Matrix")
plt.tight_layout()
plt.show()

# Save the baseline model and validation results.
Path("models").mkdir(parents=True, exist_ok=True)
Path("results").mkdir(parents=True, exist_ok=True)

joblib.dump(
    baseline_model,
    "models/tfidf_logreg_baseline.joblib"
)

metrics = {
    "experiment": "TF-IDF + Logistic Regression",
    "training_examples": len(train_clean),
    "validation_examples": len(validation_clean),
    "accuracy": float(accuracy),
    "macro_precision": float(precision),
    "macro_recall": float(recall),
    "macro_f1": float(macro_f1)
}

with open("results/baseline_validation_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("\nModel and validation metrics saved.")


In [ ]:

import json
import joblib
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Experiment 2: Same TF-IDF representation, different classifier.
svm_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        LinearSVC(
            class_weight="balanced",
            random_state=42,
            max_iter=5000
        )
    )
])

# Train only on the cleaned training split.
svm_model.fit(
    train_clean["tweet"].astype(str),
    train_clean["label"]
)

# Evaluate on validation data.
svm_predictions = svm_model.predict(
    validation_clean["tweet"].astype(str)
)

y_true = validation_clean["label"]

svm_accuracy = accuracy_score(y_true, svm_predictions)

svm_precision, svm_recall, svm_f1, _ = (
    precision_recall_fscore_support(
        y_true,
        svm_predictions,
        average="macro",
        zero_division=0
    )
)

print("EXPERIMENT 2: TF-IDF + LINEAR SVM")
print("-" * 40)
print(f"Accuracy:        {svm_accuracy:.4f}")
print(f"Macro precision: {svm_precision:.4f}")
print(f"Macro recall:    {svm_recall:.4f}")
print(f"Macro F1:        {svm_f1:.4f}")

print("\nClassification report:")
print(classification_report(
    y_true,
    svm_predictions,
    labels=["negative", "neutral", "positive"],
    target_names=["Negative", "Neutral", "Positive"],
    zero_division=0
))

# Display the confusion matrix.
cm = confusion_matrix(
    y_true,
    svm_predictions,
    labels=["negative", "neutral", "positive"]
)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Negative", "Neutral", "Positive"]
).plot(values_format="d")

plt.title("Experiment 2 — Linear SVM Validation Confusion Matrix")
plt.tight_layout()
plt.show()

# Save model and validation metrics.
Path("models").mkdir(exist_ok=True)
Path("results").mkdir(exist_ok=True)

joblib.dump(svm_model, "models/tfidf_linear_svm.joblib")

svm_metrics = {
    "experiment": "TF-IDF + Linear SVM",
    "accuracy": float(svm_accuracy),
    "macro_precision": float(svm_precision),
    "macro_recall": float(svm_recall),
    "macro_f1": float(svm_f1)
}

with open("results/linear_svm_validation_metrics.json", "w") as f:
    json.dump(svm_metrics, f, indent=2)

# Compare both models on the same validation data.
with open("results/baseline_validation_metrics.json") as f:
    baseline_metrics = json.load(f)

comparison = pd.DataFrame([
    {
        "Model": "TF-IDF + Logistic Regression",
        "Accuracy": baseline_metrics["accuracy"],
        "Macro-F1": baseline_metrics["macro_f1"]
    },
    {
        "Model": "TF-IDF + Linear SVM",
        "Accuracy": svm_accuracy,
        "Macro-F1": svm_f1
    }
])

display(comparison)
comparison.to_csv(
    "results/classical_model_comparison.csv",
    index=False
)

print("Saved model and comparison results.")


In [ ]:

from IPython.display import display as show_table

# Show the comparison table
show_table(comparison)

# Save the comparison results
comparison.to_csv(
    "results/classical_model_comparison.csv",
    index=False
)

print("\nLinear SVM validation metrics:")
print(f"Accuracy: {svm_accuracy:.4f}")
print(f"Macro-F1: {svm_f1:.4f}")

print("\nComparison saved successfully.")


In [ ]:

import json
import joblib
import pandas as pd
from pathlib import Path

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)

# Experiment 3: combine word and character TF-IDF features.
combined_model = Pipeline([
    (
        "features",
        FeatureUnion([
            (
                "word_tfidf",
                TfidfVectorizer(
                    analyzer="word",
                    ngram_range=(1, 2),
                    min_df=2,
                    sublinear_tf=True
                )
            ),
            (
                "char_tfidf",
                TfidfVectorizer(
                    analyzer="char_wb",
                    ngram_range=(3, 5),
                    min_df=2,
                    sublinear_tf=True
                )
            )
        ])
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=42
        )
    )
])

# Train on the cleaned training set only.
X_train = train_clean["tweet"].astype(str)
y_train = train_clean["label"]

combined_model.fit(X_train, y_train)

# Evaluate on validation data.
X_val = validation_clean["tweet"].astype(str)
y_val = validation_clean["label"]

combined_predictions = combined_model.predict(X_val)

combined_accuracy = accuracy_score(y_val, combined_predictions)

combined_precision, combined_recall, combined_f1, _ = (
    precision_recall_fscore_support(
        y_val,
        combined_predictions,
        average="macro",
        zero_division=0
    )
)

print("EXPERIMENT 3: WORD + CHARACTER TF-IDF")
print("-" * 42)
print(f"Accuracy:        {combined_accuracy:.4f}")
print(f"Macro precision: {combined_precision:.4f}")
print(f"Macro recall:    {combined_recall:.4f}")
print(f"Macro-F1:        {combined_f1:.4f}")

print("\nClassification report:")
print(classification_report(
    y_val,
    combined_predictions,
    labels=["negative", "neutral", "positive"],
    target_names=["Negative", "Neutral", "Positive"],
    zero_division=0
))

# Save this experiment.
Path("models").mkdir(exist_ok=True)
Path("results").mkdir(exist_ok=True)

joblib.dump(
    combined_model,
    "models/tfidf_word_char_logreg.joblib"
)

combined_metrics = {
    "experiment": "Word + Character TF-IDF + Logistic Regression",
    "accuracy": float(combined_accuracy),
    "macro_precision": float(combined_precision),
    "macro_recall": float(combined_recall),
    "macro_f1": float(combined_f1)
}

with open("results/word_char_validation_metrics.json", "w") as f:
    json.dump(combined_metrics, f, indent=2)

# Compare all three experiments on the same validation split.
with open("results/baseline_validation_metrics.json") as f:
    lr_metrics = json.load(f)

with open("results/linear_svm_validation_metrics.json") as f:
    svm_results = json.load(f)

comparison = pd.DataFrame([
    {
        "Model": "Word TF-IDF + Logistic Regression",
        "Accuracy": lr_metrics["accuracy"],
        "Macro-F1": lr_metrics["macro_f1"]
    },
    {
        "Model": "Word TF-IDF + Linear SVM",
        "Accuracy": svm_results["accuracy"],
        "Macro-F1": svm_results["macro_f1"]
    },
    {
        "Model": "Word + Character TF-IDF + Logistic Regression",
        "Accuracy": combined_accuracy,
        "Macro-F1": combined_f1
    }
])

print("\nModel comparison:")
print(comparison.to_string(index=False))

comparison.to_csv(
    "results/classical_model_comparison.csv",
    index=False
)

print("\nExperiment 3 model and metrics saved.")


In [ ]:

import json
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

# 1. Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Use CPU for dependable compatibility on your Mac.
device = torch.device("cpu")
torch.set_num_threads(2)

LABELS = ["negative", "neutral", "positive"]
label_to_id = {label: i for i, label in enumerate(LABELS)}

MAX_LEN = 80
MIN_FREQ = 2
BATCH_SIZE = 32
EPOCHS = 8

# 2. Build the vocabulary from training tweets only.
def tokenize(text):
    return str(text).lower().split()

token_counts = Counter()
for text in train_clean["tweet"].astype(str):
    token_counts.update(tokenize(text))

token_to_id = {"<pad>": 0, "<unk>": 1}

for token, count in token_counts.most_common():
    if count >= MIN_FREQ and token not in token_to_id:
        token_to_id[token] = len(token_to_id)

print("Vocabulary size:", len(token_to_id))


# 3. Convert each tweet into a sequence of token IDs.
def encode_text(text):
    ids = [
        token_to_id.get(token, token_to_id["<unk>"])
        for token in tokenize(text)
    ]
    return ids[:MAX_LEN] or [token_to_id["<unk>"]]


class SentimentDataset(Dataset):
    def __init__(self, dataframe):
        self.sequences = [
            encode_text(text)
            for text in dataframe["tweet"].astype(str)
        ]
        self.labels = [
            label_to_id[label]
            for label in dataframe["label"]
        ]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.sequences[index], self.labels[index]


def collate_batch(batch):
    sequences, labels = zip(*batch)
    lengths = torch.tensor(
        [len(sequence) for sequence in sequences],
        dtype=torch.long
    )
    max_length = int(lengths.max())

    padded = torch.zeros(
        (len(sequences), max_length),
        dtype=torch.long
    )

    for i, sequence in enumerate(sequences):
        padded[i, :len(sequence)] = torch.tensor(sequence)

    return (
        padded,
        lengths,
        torch.tensor(labels, dtype=torch.long)
    )


train_ds = SentimentDataset(train_clean)
val_ds = SentimentDataset(validation_clean)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch
)


# 4. Define a bidirectional LSTM.
class BiLSTMSentiment(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=100,
        hidden_dim=64,
        num_classes=3,
        dropout=0.3
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, input_ids, lengths):
        embeddings = self.embedding(input_ids)
        outputs, _ = self.lstm(embeddings)

        # Average only real tokens, not padding.
        positions = torch.arange(
            outputs.size(1),
            device=outputs.device
        ).unsqueeze(0)

        mask = positions < lengths.unsqueeze(1)
        mask = mask.unsqueeze(-1)

        outputs = outputs * mask
        pooled = outputs.sum(dim=1) / lengths.unsqueeze(1)

        return self.classifier(self.dropout(pooled))


model = BiLSTMSentiment(
    vocab_size=len(token_to_id)
).to(device)

# Compensate for class imbalance using training-set counts.
class_counts = train_clean["label"].value_counts()

class_weights = torch.tensor(
    [
        len(train_clean) / (len(LABELS) * class_counts[label])
        for label in LABELS
    ],
    dtype=torch.float32
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=0.0001
)


# 5. Evaluate the model on validation data.
def evaluate(model, loader):
    model.eval()
    true_ids = []
    predicted_ids = []

    with torch.no_grad():
        for inputs, lengths, labels in loader:
            logits = model(inputs.to(device), lengths.to(device))
            predictions = logits.argmax(dim=1)

            true_ids.extend(labels.numpy().tolist())
            predicted_ids.extend(predictions.cpu().numpy().tolist())

    accuracy = accuracy_score(true_ids, predicted_ids)

    precision, recall, macro_f1, _ = (
        precision_recall_fscore_support(
            true_ids,
            predicted_ids,
            labels=[0, 1, 2],
            average="macro",
            zero_division=0
        )
    )

    return (
        accuracy,
        precision,
        recall,
        macro_f1,
        true_ids,
        predicted_ids
    )


# 6. Train for up to eight epochs and select by validation macro-F1.
history = []
best_f1 = -1.0
best_state = None
best_epoch = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for inputs, lengths, labels in train_loader:
        inputs = inputs.to(device)
        lengths = lengths.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(inputs, lengths)
        loss = criterion(logits, labels)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item()

    accuracy, precision, recall, macro_f1, _, _ = evaluate(
        model, val_loader
    )

    epoch_result = {
        "epoch": epoch,
        "training_loss": running_loss / len(train_loader),
        "validation_accuracy": accuracy,
        "validation_macro_f1": macro_f1
    }
    history.append(epoch_result)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Loss: {epoch_result['training_loss']:.4f} | "
        f"Validation accuracy: {accuracy:.4f} | "
        f"Validation macro-F1: {macro_f1:.4f}"
    )

    if macro_f1 > best_f1:
        best_f1 = macro_f1
        best_epoch = epoch
        best_state = {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        }


# 7. Restore the best validation checkpoint.
model.load_state_dict(best_state)

(
    bilstm_accuracy,
    bilstm_precision,
    bilstm_recall,
    bilstm_f1,
    y_true,
    y_pred
) = evaluate(model, val_loader)

print("\nBEST BILSTM VALIDATION RESULTS")
print("Best epoch:", best_epoch)
print(f"Accuracy:        {bilstm_accuracy:.4f}")
print(f"Macro precision: {bilstm_precision:.4f}")
print(f"Macro recall:    {bilstm_recall:.4f}")
print(f"Macro-F1:        {bilstm_f1:.4f}")

print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1, 2],
    target_names=["Negative", "Neutral", "Positive"],
    zero_division=0
))


# 8. Plot the confusion matrix.
cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Negative", "Neutral", "Positive"]
).plot(values_format="d")

plt.title("Experiment 4 — BiLSTM Validation Confusion Matrix")
plt.tight_layout()
plt.show()


# 9. Save the model, vocabulary, training history, and metrics.
Path("models").mkdir(exist_ok=True)
Path("results").mkdir(exist_ok=True)

torch.save(
    {
        "model_state_dict": best_state,
        "vocabulary": token_to_id,
        "label_to_id": label_to_id,
        "max_len": MAX_LEN,
        "best_epoch": best_epoch,
        "embedding_dim": 100,
        "hidden_dim": 64
    },
    "models/bilstm_kinyarwanda.pt"
)

pd.DataFrame(history).to_csv(
    "results/bilstm_training_history.csv",
    index=False
)

bilstm_metrics = {
    "experiment": "BiLSTM",
    "best_epoch": int(best_epoch),
    "accuracy": float(bilstm_accuracy),
    "macro_precision": float(bilstm_precision),
    "macro_recall": float(bilstm_recall),
    "macro_f1": float(bilstm_f1)
}

with open("results/bilstm_validation_metrics.json", "w") as f:
    json.dump(bilstm_metrics, f, indent=2)

print("\nBiLSTM checkpoint, history, and metrics saved.")


In [ ]:

import sys
import importlib.util
import torch
import platform

print("Python version:", sys.version.split()[0])
print("Operating system:", platform.platform())
print("PyTorch version:", torch.__version__)

print("\nAvailable hardware:")
print("Apple Silicon MPS available:", torch.backends.mps.is_available())
print("CUDA available:", torch.cuda.is_available())

print("\nRequired libraries:")
print("Transformers installed:", importlib.util.find_spec("transformers") is not None)
print("SentencePiece installed:", importlib.util.find_spec("sentencepiece") is not None)

try:
    import psutil
    ram_gb = psutil.virtual_memory().total / (1024 ** 3)
    print(f"Total system RAM: {ram_gb:.1f} GB")
except ImportError:
    print("System RAM information unavailable.")


In [ ]:
%pip install sentencepiece

In [ ]:
import sentencepiece

print("SentencePiece version:", sentencepiece.__version__)
print("Ready for the Transformer tokenizer.")

In [ ]:

from transformers import AutoTokenizer

MODEL_NAME = "Davlan/afro-xlmr-base"

print("Loading AfroXLM-R tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Use a real tweet from our cleaned training dataset.
sample_text = str(train_clean.iloc[0]["tweet"])
encoded = tokenizer(
    sample_text,
    truncation=True,
    max_length=128
)

print("\nTokenizer loaded successfully!")
print("Vocabulary size:", tokenizer.vocab_size)
print("Model input length:", len(encoded["input_ids"]))
print("Sample tweet:", sample_text)
print("Token IDs:", encoded["input_ids"][:20])
print("Decoded tokens:", tokenizer.convert_ids_to_tokens(
    encoded["input_ids"][:20]
))


In [ ]:

import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

# --------------------------------------------------
# 1. Reproducibility and configuration
# --------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

LABELS = ["negative", "neutral", "positive"]
LABEL_TO_ID = {label: i for i, label in enumerate(LABELS)}
ID_TO_LABEL = {i: label for i, label in enumerate(LABELS)}

MODEL_NAME = "Davlan/afro-xlmr-base"
MAX_LENGTH = 128
BATCH_SIZE = 4
GRAD_ACCUMULATION = 4
EPOCHS = 3
LEARNING_RATE = 2e-5

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

print("Device:", device)
print("Training examples:", len(train_clean))
print("Validation examples:", len(validation_clean))

# Check that labels match the expected three classes.
assert set(train_clean["label"]).issubset(LABELS)
assert set(validation_clean["label"]).issubset(LABELS)

# --------------------------------------------------
# 2. Load tokenizer and pretrained model
# --------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=ID_TO_LABEL,
    label2id=LABEL_TO_ID
)

model.to(device)

print("Pretrained Transformer loaded.")
print("Device used:", device)

# --------------------------------------------------
# 3. Tokenize the tweets
# --------------------------------------------------
train_encodings = tokenizer(
    train_clean["tweet"].astype(str).tolist(),
    truncation=True,
    max_length=MAX_LENGTH
)

validation_encodings = tokenizer(
    validation_clean["tweet"].astype(str).tolist(),
    truncation=True,
    max_length=MAX_LENGTH
)


class EncodedSentimentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = [
            LABEL_TO_ID[label] for label in labels
        ]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            key: value[idx]
            for key, value in self.encodings.items()
        }
        item["labels"] = self.labels[idx]
        return item


def collate_batch(features):
    labels = torch.tensor(
        [feature["labels"] for feature in features],
        dtype=torch.long
    )

    inputs = [
        {
            key: value
            for key, value in feature.items()
            if key != "labels"
        }
        for feature in features
    ]

    batch = tokenizer.pad(
        inputs,
        padding=True,
        return_tensors="pt"
    )
    batch["labels"] = labels
    return batch


train_dataset = EncodedSentimentDataset(
    train_encodings,
    train_clean["label"].tolist()
)

validation_dataset = EncodedSentimentDataset(
    validation_encodings,
    validation_clean["label"].tolist()
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch,
    num_workers=0
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch,
    num_workers=0
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))

# --------------------------------------------------
# 4. Configure training
# --------------------------------------------------
class_counts = train_clean["label"].value_counts()

class_weights = torch.tensor(
    [
        len(train_clean) / (
            len(LABELS) * class_counts[label]
        )
        for label in LABELS
    ],
    dtype=torch.float32,
    device=device
)

criterion = torch.nn.CrossEntropyLoss(
    weight=class_weights
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.01
)


# --------------------------------------------------
# 5. Validation function
# --------------------------------------------------
def evaluate_transformer(model, loader):
    model.eval()
    true_ids = []
    predicted_ids = []

    with torch.no_grad():
        for batch in loader:
            labels = batch["labels"]

            inputs = {
                key: value.to(device)
                for key, value in batch.items()
                if key != "labels"
            }

            outputs = model(**inputs)
            predictions = outputs.logits.argmax(dim=1)

            true_ids.extend(labels.numpy().tolist())
            predicted_ids.extend(
                predictions.cpu().numpy().tolist()
            )

    accuracy = accuracy_score(
        true_ids, predicted_ids
    )

    precision, recall, macro_f1, _ = (
        precision_recall_fscore_support(
            true_ids,
            predicted_ids,
            labels=[0, 1, 2],
            average="macro",
            zero_division=0
        )
    )

    return (
        accuracy,
        precision,
        recall,
        macro_f1,
        true_ids,
        predicted_ids
    )


# --------------------------------------------------
# 6. Fine-tune and select the best validation checkpoint
# --------------------------------------------------
Path("models").mkdir(exist_ok=True)
Path("results").mkdir(exist_ok=True)

BEST_DIR = Path("models/afroxlmr_best")
history = []
best_f1 = -1.0
best_epoch = None
best_metrics = None

optimizer.zero_grad(set_to_none=True)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0

    for step, batch in enumerate(train_loader):
        labels = batch["labels"].to(device)

        inputs = {
            key: value.to(device)
            for key, value in batch.items()
            if key != "labels"
        }

        outputs = model(**inputs)
        loss = criterion(outputs.logits, labels)

        (loss / GRAD_ACCUMULATION).backward()
        total_loss += loss.item()

        is_update_step = (
            (step + 1) % GRAD_ACCUMULATION == 0
            or (step + 1) == len(train_loader)
        )

        if is_update_step:
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        if (step + 1) % 100 == 0:
            print(
                f"Epoch {epoch}/{EPOCHS} — "
                f"batch {step + 1}/{len(train_loader)}",
                flush=True
            )

    (
        val_accuracy,
        val_precision,
        val_recall,
        val_f1,
        y_true,
        y_pred
    ) = evaluate_transformer(model, validation_loader)

    epoch_metrics = {
        "epoch": epoch,
        "training_loss": total_loss / len(train_loader),
        "validation_accuracy": float(val_accuracy),
        "validation_macro_precision": float(val_precision),
        "validation_macro_recall": float(val_recall),
        "validation_macro_f1": float(val_f1)
    }

    history.append(epoch_metrics)

    print("\nEpoch results:", epoch_metrics, flush=True)

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_epoch = epoch
        best_metrics = dict(epoch_metrics)

        # Save the best checkpoint, not just the final epoch.
        model.save_pretrained(
            BEST_DIR,
            safe_serialization=True
        )
        tokenizer.save_pretrained(BEST_DIR)

        print(
            f"New best checkpoint saved: epoch {epoch}, "
            f"macro-F1={val_f1:.4f}",
            flush=True
        )

    print("-" * 50, flush=True)


# --------------------------------------------------
# 7. Report the best validation results
# --------------------------------------------------
print("\nBEST AFROXLM-R VALIDATION RESULTS")
print("Best epoch:", best_epoch)
print(f"Accuracy:        {best_metrics['validation_accuracy']:.4f}")
print(f"Macro precision: {best_metrics['validation_macro_precision']:.4f}")
print(f"Macro recall:    {best_metrics['validation_macro_recall']:.4f}")
print(f"Macro-F1:        {best_metrics['validation_macro_f1']:.4f}")

print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1, 2],
    target_names=["Negative", "Neutral", "Positive"],
    zero_division=0
))

ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1, 2]
    ),
    display_labels=["Negative", "Neutral", "Positive"]
).plot(values_format="d")

plt.title("AfroXLM-R — Validation Confusion Matrix")
plt.tight_layout()
plt.show()

pd.DataFrame(history).to_csv(
    "results/afroxlmr_training_history.csv",
    index=False
)

with open("results/afroxlmr_validation_metrics.json", "w") as f:
    json.dump(best_metrics, f, indent=2)

print("\nBest checkpoint saved in:", BEST_DIR)
print("Training history and validation metrics saved.")


In [ ]:

# Reload the best saved AfroXLM-R checkpoint.
from transformers import AutoModelForSequenceClassification
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
import json
import matplotlib.pyplot as plt

best_model = AutoModelForSequenceClassification.from_pretrained(
    "models/afroxlmr_best"
).to(device)

# Re-evaluate that exact checkpoint on validation data.
(
    best_accuracy,
    best_precision,
    best_recall,
    best_f1,
    best_true_ids,
    best_predicted_ids
) = evaluate_transformer(best_model, validation_loader)

print("BEST CHECKPOINT — VALIDATION RESULTS")
print(f"Accuracy:        {best_accuracy:.4f}")
print(f"Macro precision: {best_precision:.4f}")
print(f"Macro recall:    {best_recall:.4f}")
print(f"Macro-F1:        {best_f1:.4f}")

print("\nClassification report:")
print(classification_report(
    best_true_ids,
    best_predicted_ids,
    labels=[0, 1, 2],
    target_names=["Negative", "Neutral", "Positive"],
    zero_division=0
))

ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(
        best_true_ids,
        best_predicted_ids,
        labels=[0, 1, 2]
    ),
    display_labels=["Negative", "Neutral", "Positive"]
).plot(values_format="d")

plt.title("Best AfroXLM-R Checkpoint — Validation")
plt.tight_layout()
plt.show()

# Save the consistent metrics for the report.
corrected_metrics = {
    "experiment": "AfroXLM-R",
    "best_epoch": best_epoch,
    "accuracy": float(best_accuracy),
    "macro_precision": float(best_precision),
    "macro_recall": float(best_recall),
    "macro_f1": float(best_f1)
}

with open("results/afroxlmr_validation_metrics.json", "w") as f:
    json.dump(corrected_metrics, f, indent=2)

print("\nBest checkpoint and validation metrics verified.")


In [ ]:

import json
from pathlib import Path
import pandas as pd

results_dir = Path("results")

experiment_files = {
    "TF-IDF + Logistic Regression":
        "baseline_validation_metrics.json",
    "TF-IDF + Linear SVM":
        "linear_svm_validation_metrics.json",
    "Word + Character TF-IDF + Logistic Regression":
        "word_char_validation_metrics.json",
    "BiLSTM":
        "bilstm_validation_metrics.json",
    "AfroXLM-R":
        "afroxlmr_validation_metrics.json",
}

rows = []

for model_name, filename in experiment_files.items():
    filepath = results_dir / filename

    if not filepath.exists():
        print(f"Missing results file: {filepath}")
        continue

    with open(filepath, "r") as file:
        metrics = json.load(file)

    rows.append({
        "Model": model_name,
        "Accuracy": metrics.get("accuracy"),
        "Macro Precision": metrics.get("macro_precision"),
        "Macro Recall": metrics.get("macro_recall"),
        "Macro-F1": metrics.get("macro_f1"),
    })

results_table = pd.DataFrame(rows)

results_table = results_table.sort_values(
    by="Macro-F1",
    ascending=False
).reset_index(drop=True)

print("VALIDATION RESULTS — ALL EXPERIMENTS")
print(results_table.to_string(
    index=False,
    formatters={
        "Accuracy": lambda x: f"{x:.4f}",
        "Macro Precision": lambda x: f"{x:.4f}",
        "Macro Recall": lambda x: f"{x:.4f}",
        "Macro-F1": lambda x: f"{x:.4f}",
    }
))

results_table.to_csv(
    results_dir / "validation_results_summary.csv",
    index=False
)

print("\nSaved results/validation_results_summary.csv")


In [ ]:

import gc
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

# Free memory used by the previous Transformer models.
if "model" in globals():
    del model
if "best_model" in globals():
    del best_model

gc.collect()

if torch.backends.mps.is_available():
    torch.mps.empty_cache()

# Reproducible initialization.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    torch.mps.manual_seed(SEED)

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

# Only the learning rate changes from the previous experiment.
NEW_LR = 1e-5
EPOCHS_LR = 3
GRAD_ACCUMULATION_LR = 4

print("Device:", device)
print("Learning rate:", NEW_LR)
print("Training examples:", len(train_clean))
print("Validation examples:", len(validation_clean))

# Start from the original pretrained checkpoint.
model_lr = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=ID_TO_LABEL,
    label2id=LABEL_TO_ID
).to(device)

# Same class weights as the original experiment.
class_counts_lr = train_clean["label"].value_counts()

class_weights_lr = torch.tensor(
    [
        len(train_clean) / (
            len(LABELS) * class_counts_lr[label]
        )
        for label in LABELS
    ],
    dtype=torch.float32,
    device=device
)

criterion_lr = torch.nn.CrossEntropyLoss(
    weight=class_weights_lr
)

optimizer_lr = torch.optim.AdamW(
    model_lr.parameters(),
    lr=NEW_LR,
    weight_decay=0.01
)

output_dir_lr = Path("models/afroxlmr_lr1e5_best")
results_dir_lr = Path("results")

output_dir_lr.mkdir(parents=True, exist_ok=True)
results_dir_lr.mkdir(parents=True, exist_ok=True)

history_lr = []
best_f1_lr = -1.0
best_epoch_lr = None
best_metrics_lr = None
best_true_lr = None
best_pred_lr = None

optimizer_lr.zero_grad(set_to_none=True)

# Train for up to three epochs.
for epoch in range(1, EPOCHS_LR + 1):
    model_lr.train()
    running_loss_lr = 0.0

    for step, batch in enumerate(train_loader):
        labels = batch["labels"].to(device)

        inputs = {
            key: value.to(device)
            for key, value in batch.items()
            if key != "labels"
        }

        outputs = model_lr(**inputs)
        loss = criterion_lr(outputs.logits, labels)

        (loss / GRAD_ACCUMULATION_LR).backward()
        running_loss_lr += loss.item()

        should_update = (
            (step + 1) % GRAD_ACCUMULATION_LR == 0
            or (step + 1) == len(train_loader)
        )

        if should_update:
            torch.nn.utils.clip_grad_norm_(
                model_lr.parameters(),
                max_norm=1.0
            )
            optimizer_lr.step()
            optimizer_lr.zero_grad(set_to_none=True)

        if (step + 1) % 100 == 0:
            print(
                f"Epoch {epoch}/{EPOCHS_LR} — "
                f"batch {step + 1}/{len(train_loader)}",
                flush=True
            )

    (
        val_accuracy_lr,
        val_precision_lr,
        val_recall_lr,
        val_f1_lr,
        true_ids_lr,
        pred_ids_lr
    ) = evaluate_transformer(model_lr, validation_loader)

    epoch_metrics_lr = {
        "epoch": epoch,
        "learning_rate": NEW_LR,
        "training_loss": running_loss_lr / len(train_loader),
        "accuracy": float(val_accuracy_lr),
        "macro_precision": float(val_precision_lr),
        "macro_recall": float(val_recall_lr),
        "macro_f1": float(val_f1_lr)
    }

    history_lr.append(epoch_metrics_lr)
    print("\nEpoch results:", epoch_metrics_lr, flush=True)

    # Save the checkpoint with the best validation macro-F1.
    if val_f1_lr > best_f1_lr:
        best_f1_lr = val_f1_lr
        best_epoch_lr = epoch
        best_metrics_lr = dict(epoch_metrics_lr)
        best_true_lr = list(true_ids_lr)
        best_pred_lr = list(pred_ids_lr)

        model_lr.save_pretrained(
            output_dir_lr,
            safe_serialization=True
        )
        tokenizer.save_pretrained(output_dir_lr)

        print(
            f"New best checkpoint: epoch {epoch}, "
            f"macro-F1={val_f1_lr:.4f}",
            flush=True
        )

    print("-" * 50, flush=True)

# Report the selected checkpoint.
print("\nAFROXLM-R — LEARNING RATE 1e-5")
print("Best epoch:", best_epoch_lr)
print(f"Accuracy:        {best_metrics_lr['accuracy']:.4f}")
print(f"Macro precision: {best_metrics_lr['macro_precision']:.4f}")
print(f"Macro recall:    {best_metrics_lr['macro_recall']:.4f}")
print(f"Macro-F1:        {best_metrics_lr['macro_f1']:.4f}")

print("\nClassification report:")
print(classification_report(
    best_true_lr,
    best_pred_lr,
    labels=[0, 1, 2],
    target_names=["Negative", "Neutral", "Positive"],
    zero_division=0
))

ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(
        best_true_lr,
        best_pred_lr,
        labels=[0, 1, 2]
    ),
    display_labels=["Negative", "Neutral", "Positive"]
).plot(values_format="d")

plt.title("AfroXLM-R — Learning Rate 1e-5")
plt.tight_layout()
plt.show()

# Save this experiment separately.
pd.DataFrame(history_lr).to_csv(
    results_dir_lr / "afroxlmr_lr1e5_history.csv",
    index=False
)

with open(
    results_dir_lr / "afroxlmr_lr1e5_validation_metrics.json",
    "w"
) as f:
    json.dump(best_metrics_lr, f, indent=2)

print("\nCheckpoint and results saved.")


In [ ]:

import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# --------------------------------------------------
# 1. Load the selected model and its tokenizer
# --------------------------------------------------
BEST_MODEL_DIR = "models/afroxlmr_lr1e5_best"

final_tokenizer = AutoTokenizer.from_pretrained(BEST_MODEL_DIR)

final_model = AutoModelForSequenceClassification.from_pretrained(
    BEST_MODEL_DIR
).to(device)

# Update the tokenizer used by the existing collate function.
tokenizer = final_tokenizer

print("Selected model loaded.")
print("Test examples:", len(test_clean))

# --------------------------------------------------
# 2. Prepare the held-out test data
# --------------------------------------------------
test_encodings = tokenizer(
    test_clean["tweet"].astype(str).tolist(),
    truncation=True,
    max_length=MAX_LENGTH
)

test_dataset = EncodedSentimentDataset(
    test_encodings,
    test_clean["label"].tolist()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch,
    num_workers=0
)

# --------------------------------------------------
# 3. Evaluate exactly once on the test set
# --------------------------------------------------
(
    test_accuracy,
    test_precision,
    test_recall,
    test_macro_f1,
    test_true_ids,
    test_predicted_ids
) = evaluate_transformer(final_model, test_loader)

print("\nFINAL HELD-OUT TEST RESULTS")
print("-" * 40)
print(f"Accuracy:        {test_accuracy:.4f}")
print(f"Macro precision: {test_precision:.4f}")
print(f"Macro recall:    {test_recall:.4f}")
print(f"Macro-F1:        {test_macro_f1:.4f}")

print("\nClassification report:")
print(classification_report(
    test_true_ids,
    test_predicted_ids,
    labels=[0, 1, 2],
    target_names=["Negative", "Neutral", "Positive"],
    zero_division=0
))

# --------------------------------------------------
# 4. Confusion matrix
# --------------------------------------------------
test_cm = confusion_matrix(
    test_true_ids,
    test_predicted_ids,
    labels=[0, 1, 2]
)

ConfusionMatrixDisplay(
    confusion_matrix=test_cm,
    display_labels=["Negative", "Neutral", "Positive"]
).plot(values_format="d")

plt.title("AfroXLM-R — Final Test Confusion Matrix")
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 5. Save the final test metrics
# --------------------------------------------------
final_test_metrics = {
    "model": "Davlan/afro-xlmr-base",
    "learning_rate": 1e-5,
    "best_validation_epoch": int(best_epoch_lr),
    "test_examples": len(test_clean),
    "accuracy": float(test_accuracy),
    "macro_precision": float(test_precision),
    "macro_recall": float(test_recall),
    "macro_f1": float(test_macro_f1)
}

Path("results").mkdir(exist_ok=True)

with open("results/final_test_metrics.json", "w") as f:
    json.dump(final_test_metrics, f, indent=2)

print("\nFinal test metrics saved to results/final_test_metrics.json")


In [ ]:

from pathlib import Path
import pandas as pd
from IPython.display import display as show_table

# Associate test tweets with the actual and predicted labels.
error_analysis = pd.DataFrame({
    "tweet": test_clean["tweet"].astype(str).reset_index(drop=True),
    "true_label": [
        ID_TO_LABEL[i] for i in test_true_ids
    ],
    "predicted_label": [
        ID_TO_LABEL[i] for i in test_predicted_ids
    ]
})

error_analysis["correct"] = (
    error_analysis["true_label"]
    == error_analysis["predicted_label"]
)

# Keep only the incorrect predictions for detailed inspection.
misclassified = error_analysis[
    ~error_analysis["correct"]
].copy()

misclassified["error_pair"] = (
    misclassified["true_label"]
    + " → "
    + misclassified["predicted_label"]
)

print("TEST ERROR ANALYSIS")
print("-" * 40)
print("Total test tweets:", len(error_analysis))
print("Correct predictions:", int(error_analysis["correct"].sum()))
print("Incorrect predictions:", len(misclassified))
print(
    "Observed error rate:",
    f"{len(misclassified) / len(error_analysis):.2%}"
)

print("\nMost frequent error patterns:")
print(misclassified["error_pair"].value_counts().to_string())

print("\nExamples of incorrectly classified tweets:")
show_table(
    misclassified[
        ["tweet", "true_label", "predicted_label", "error_pair"]
    ].head(15)
)

# Save the analysis for the report.
Path("results").mkdir(exist_ok=True)

error_analysis.to_csv(
    "results/test_predictions_all.csv",
    index=False
)

misclassified.to_csv(
    "results/test_misclassified_examples.csv",
    index=False
)

misclassified["error_pair"].value_counts().to_csv(
    "results/test_error_pattern_counts.csv",
    header=["count"]
)

print("\nSaved:")
print("- results/test_predictions_all.csv")
print("- results/test_misclassified_examples.csv")
print("- results/test_error_pattern_counts.csv")

## Preprocessing decision log

Do not aggressively remove punctuation, emojis, hashtags, mentions, or stopwords before the baseline. Social-media sentiment can depend on these signals. We will compare targeted cleaning choices as experiments rather than silently applying them.